## RAG Ingestion Pipeline

Ingestion ==>  Chunking ==> Embedding ==> Vector Store

In [15]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [16]:
def process_all_pdfs(pdf_directory):
    all_documents = []

    pdf_files = list(filter(lambda f: f.endswith('.pdf'), os.listdir(pdf_directory)))
    print(f"Found {len(pdf_files)} PDF files in {pdf_directory}.")


    for filename in pdf_files:

        pdf_path = os.path.join(pdf_directory, filename)
        print(f"\n Processing {filename} with {len(filename)} page(s).")

        try:
            loader = PyPDFLoader(pdf_path)
            documents = loader.load()

            for doc in documents:
                doc.metadata["source"] = filename
                doc.metadata["total_pages"] = len(documents)
                doc.metadata["file_size"] = os.path.getsize(pdf_path)
                doc.metadata["file_type"] = "pdf"

            all_documents.extend(documents)
            print(f"Successfully loaded {len(documents)} page(s) from {filename} using PyPDFLoader.")

        except Exception as e:
            print(f"Failed to load {filename} with PyPDFLoader: {e}. ")

    print(f"\n Total documents loaded: {len(all_documents)}")
    return all_documents

            
all_pdf_documents = process_all_pdfs("../data/pdf_files")

Found 11 PDF files in ../data/pdf_files.

 Processing acceptable_use_policy.pdf with 25 page(s).
Successfully loaded 1 page(s) from acceptable_use_policy.pdf using PyPDFLoader.

 Processing business_continuity_policy.pdf with 30 page(s).
Successfully loaded 1 page(s) from business_continuity_policy.pdf using PyPDFLoader.

 Processing code_of_conduct.pdf with 19 page(s).
Successfully loaded 1 page(s) from code_of_conduct.pdf using PyPDFLoader.

 Processing company_overview.pdf with 20 page(s).
Successfully loaded 1 page(s) from company_overview.pdf using PyPDFLoader.

 Processing diversity_equity_inclusion_policy.pdf with 37 page(s).
Successfully loaded 1 page(s) from diversity_equity_inclusion_policy.pdf using PyPDFLoader.

 Processing hr_policies_handbook.pdf with 24 page(s).
Successfully loaded 1 page(s) from hr_policies_handbook.pdf using PyPDFLoader.

 Processing incident_response_policy.pdf with 28 page(s).
Successfully loaded 1 page(s) from incident_response_policy.pdf using PyPD

In [17]:
all_pdf_documents

[Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': 'acceptable_use_policy.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'file_size': 2390, 'file_type': 'pdf'}, page_content='Northstar Meridian Group\nAcceptable Use Policy\nDocument ID: IT-AUP-006   |   Owner: IT Operations   |   Version: 2.0   |   Effective: 2026-07-26\nCompany devices and accounts are provided for business use.\nUsers must:\n- Keep devices updated and locked when unattended\n- Use only approved software and cloud services\n- Avoid installing unlicensed or malicious software\n- Store company files in approved repositories\n- Report lost devices, phishing, or suspicious activity promptly\nUsers must not:\n- Circumvent security controls or monitoring\n- Share corporate accounts or bypass access reviews\n- Download copyright-protected material without permission\n- Use resources for gambling, harassment, or other inappropriate content\nInternal use only | Internal Knowledge 

In [18]:
## Text splitting into chunks 
def split_documents(documents, chunk_size=100, chunk_overlap=20):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks.")

    return split_docs

In [19]:
chunks = split_documents(all_pdf_documents)
chunks

Split 11 documents into 129 chunks.


[Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': 'acceptable_use_policy.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'file_size': 2390, 'file_type': 'pdf'}, page_content='Northstar Meridian Group\nAcceptable Use Policy'),
 Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': 'acceptable_use_policy.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'file_size': 2390, 'file_type': 'pdf'}, page_content='Document ID: IT-AUP-006   |   Owner: IT Operations   |   Version: 2.0   |   Effective: 2026-07-26'),
 Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': 'acceptable_use_policy.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'file_size': 2390, 'file_type': 'pdf'}, page_content='Company devices and accounts are provided for business use.\nUsers must:'),
 Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': 'acceptable_use

## Embedding and Vector Store BD

In [21]:
import numpy as np
import chromadb
from sentence_transformers import SentenceTransformer
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

ModuleNotFoundError: No module named 'chromadb'